In [2]:
import pandas as pd
import numpy as np
import json
import random
import os
import time

# ==========================================
# 1. KONFIGURASI PARAMETER GA & VRP
# ==========================================
POP_SIZE = 50
GENERATIONS = 100
CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.2
MAX_WORK_MINUTES = 600  # 10 jam (batas waktu per kurir)
SERVICE_TIME = 20       # 20 menit waktu pelayanan per puskesmas

NUM_RUNS = 10 # Jumlah iterasi untuk setiap klaster

KLASTER_LIST = ['barat', 'pusat', 'selatan', 'timur', 'utara']
PATH_DATA = r'../data/'
PATH_JSON = r'../output_json/'

if not os.path.exists(PATH_JSON):
    os.makedirs(PATH_JSON)

In [3]:
# ==========================================
# 2. FUNGSI PENDUKUNG (DATA & PRIORITAS)
# ==========================================
def bersihkan_kordinat(val):
    if pd.isna(val): return 0.0
    val_str = str(val).replace(',', '').replace('.', '').replace(' ', '')
    if '-' in str(val):
        angka = val_str.replace('-', '')
        return float('-' + angka[:1] + '.' + angka[1:]) if len(angka) > 1 else float(val)
    else:
        return float(val_str[:3] + '.' + val_str[3:]) if len(val_str) > 3 else float(val)

def penentuan_prioritas(jaringan, layanan):
    jaringan = str(jaringan).lower()
    layanan = str(layanan).lower()
    
    # 1: Induk & Inap, 2: Induk & Jalan, 3: Pustu/Pembantu & Jalan
    if 'induk' in jaringan and 'inap' in layanan: return 1
    elif 'induk' in jaringan and 'jalan' in layanan: return 2
    else: return 3

In [4]:
# ==========================================
# 3. KELAS GENETIC ALGORITHM VRP
# ==========================================
class GAVRP:
    def __init__(self, klaster):
        self.klaster = klaster
        self.load_data()
        
    def load_data(self):
        # Load Matriks Jarak dan Waktu
        self.df_jarak = pd.read_csv(f"{PATH_DATA}matriks_jarak_riil_{self.klaster}.csv", index_col=0)
        self.df_waktu = pd.read_csv(f"{PATH_DATA}datamatriks_waktu_{self.klaster}.csv", index_col=0)
        
        self.nodes = list(self.df_jarak.columns)
        self.num_nodes = len(self.nodes)
        
        # Load Koordinat & Atribut untuk Prioritas
        df_all = pd.read_csv(f"{PATH_DATA}koordinat_eas.csv")
        df_all['Latitude'] = df_all['Latitude'].apply(bersihkan_kordinat)
        df_all['Longitude'] = df_all['Longitude'].apply(bersihkan_kordinat)
        
        self.node_info = {}
        for node in self.nodes:
            row = df_all[df_all['Nama Puskesmas'].str.strip() == node.strip()]
            if not row.empty:
                row = row.iloc[0]
                prio = penentuan_prioritas(row['Jaringan Pelayanan'], row['Jenis Layanan'])
                lat, lon = row['Latitude'], row['Longitude']
            else:
                prio = 1 if 'UPTD' in node else 3
                lat, lon = 0.0, 0.0
                
            self.node_info[node] = {'prioritas': prio, 'koordinat': [lat, lon]}

    def init_population(self):
        population = []
        base_tour = list(range(1, self.num_nodes))
        # Sort heuristik awal: prioritas VIP ditaruh di depan array
        base_tour.sort(key=lambda x: self.node_info[self.nodes[x]]['prioritas'])
        
        for _ in range(POP_SIZE):
            indv = base_tour.copy()
            for _ in range(len(indv) // 3):
                a, b = random.sample(range(len(indv)), 2)
                # Swap aman: hanya menukar elemen yang level prioritasnya berdekatan
                if abs(self.node_info[self.nodes[indv[a]]]['prioritas'] - self.node_info[self.nodes[indv[b]]]['prioritas']) <= 1:
                    indv[a], indv[b] = indv[b], indv[a]
            population.append(indv)
        return population

    def calculate_fitness(self, chromosome):
        routes = []
        current_route = []
        current_time = 0.0
        current_dist = 0.0
        curr_node = 0
        total_time_all = 0.0
        total_dist_all = 0.0
        penalty = 0
        
        # Penalti Urutan - Pustu tidak boleh didatangi sebelum VIP
        for i in range(len(chromosome)):
            for j in range(i+1, len(chromosome)):
                prio_awal = self.node_info[self.nodes[chromosome[i]]]['prioritas']
                prio_akhir = self.node_info[self.nodes[chromosome[j]]]['prioritas']
                if prio_awal > prio_akhir:
                    penalty += 50 

        # Splitting Rute Otomatis (Constraint Maksimal 10 Jam)
        for node in chromosome:
            waktu_jalan = self.df_waktu.iloc[curr_node, node]
            waktu_pulang = self.df_waktu.iloc[node, 0]
            
            # Cek simulasi: Jika perjalanan + ngurus obat + pulang melebihi 10 jam?
            estimasi_waktu = current_time + waktu_jalan + SERVICE_TIME + waktu_pulang
            
            if estimasi_waktu > MAX_WORK_MINUTES and len(current_route) > 0:
                # Waktu habis, kurir pulang ke UPTD (Node 0)
                current_time += self.df_waktu.iloc[curr_node, 0]
                current_dist += self.df_jarak.iloc[curr_node, 0]
                routes.append({'rute': current_route, 'waktu': current_time, 'jarak': current_dist})
                
                total_time_all += current_time
                total_dist_all += current_dist
                
                # Buka tugas untuk kurir baru
                current_route = [node]
                current_time = self.df_waktu.iloc[0, node] + SERVICE_TIME
                current_dist = self.df_jarak.iloc[0, node]
                curr_node = node
            else:
                # Waktu masih cukup, lanjut tambah kunjungan
                current_route.append(node)
                current_time += waktu_jalan + SERVICE_TIME
                current_dist += self.df_jarak.iloc[curr_node, node]
                curr_node = node
                
        # Tutup rute kurir yang paling terakhir
        current_time += self.df_waktu.iloc[curr_node, 0]
        current_dist += self.df_jarak.iloc[curr_node, 0]
        routes.append({'rute': current_route, 'waktu': current_time, 'jarak': current_dist})
        
        total_time_all += current_time
        total_dist_all += current_dist
        
        fitness_score = total_dist_all + penalty
        return fitness_score, routes, total_dist_all, total_time_all

    def crossover(self, parent1, parent2):
        if random.random() > CROSSOVER_RATE: return parent1.copy()
        start, end = sorted(random.sample(range(len(parent1)), 2))
        child = [None] * len(parent1)
        child[start:end+1] = parent1[start:end+1]
        p2_idx = 0
        for i in range(len(child)):
            if child[i] is None:
                while parent2[p2_idx] in child: p2_idx += 1
                child[i] = parent2[p2_idx]
        return child

    def mutate(self, indv):
        if random.random() < MUTATION_RATE:
            a, b = random.sample(range(len(indv)), 2)
            indv[a], indv[b] = indv[b], indv[a]
        return indv

    # Fungsi pengeksekusi algoritma untuk 1x run
    def run_single(self):
        waktu_mulai = time.time()
        pop = self.init_population()
        best_fitness = float('inf')
        best_routes_info = None
        riwayat_konvergensi = []
        
        for gen in range(GENERATIONS):
            pop_fitness = []
            for indv in pop:
                fit, routes, t_dist, t_waktu = self.calculate_fitness(indv)
                pop_fitness.append((fit, indv, routes, t_dist, t_waktu))
                
                if fit < best_fitness:
                    best_fitness = fit
                    best_routes_info = (routes, t_dist, t_waktu)
            
            riwayat_konvergensi.append(round(best_routes_info[1], 3))
            
            pop_fitness.sort(key=lambda x: x[0])
            new_pop = [pop_fitness[0][1], pop_fitness[1][1]]
            while len(new_pop) < POP_SIZE:
                p1 = random.choice(pop_fitness[:20])[1]
                p2 = random.choice(pop_fitness[:20])[1]
                child = self.crossover(p1, p2)
                child = self.mutate(child)
                new_pop.append(child)
            pop = new_pop
            
        runtime_detik = round(time.time() - waktu_mulai, 3)
        return {
            "fitness": best_fitness,
            "rute_mentah": best_routes_info[0],
            "total_jarak": best_routes_info[1],
            "total_waktu": best_routes_info[2],
            "riwayat_konvergensi": riwayat_konvergensi,
            "runtime_detik": runtime_detik
        }

    # Fungsi utama untuk melakukan looping 10x run dan mengukur statistik
    def run_multiple(self, num_runs=10):
        all_runs = []
        best_overall_run = None
        best_overall_fitness = float('inf')
        
        for i in range(1, num_runs + 1):
            run_result = self.run_single()
            all_runs.append(run_result)
            
            # Cari run terbaik berdasarkan nilai fitness terkecil (Minimum)
            if run_result['fitness'] < best_overall_fitness:
                best_overall_fitness = run_result['fitness']
                best_overall_run = run_result
                
            print(f"      ➔ Run {i}/{num_runs} Selesai | Fitness: {round(run_result['fitness'], 2)} | Waktu: {run_result['runtime_detik']} dtk")
            
        # Ekstrak data fitness dan runtime untuk kebutuhan analisis deskriptif
        semua_fitness = [round(r['fitness'], 2) for r in all_runs]
        semua_runtime = [r['runtime_detik'] for r in all_runs]
        
        fit_min = round(np.min(semua_fitness), 2)
        fit_mean = round(np.mean(semua_fitness), 2)
        fit_std = round(np.std(semua_fitness, ddof=1) if num_runs > 1 else 0.0, 2)
        runtime_mean = round(np.mean(semua_runtime), 3)
        
        return self.format_final_json(best_overall_run, semua_fitness, fit_min, fit_mean, fit_std, runtime_mean)

    def format_final_json(self, best_run, semua_fitness, fit_min, fit_mean, fit_std, runtime_mean):
        nama_depot = self.nodes[0]
        klaster_data = {
            "statistik_10_run": {
                "fitness_minimum": fit_min,
                "fitness_rata_rata": fit_mean,
                "fitness_std_dev": fit_std,
                "waktu_komputasi_rata_rata_detik": runtime_mean,
                "semua_fitness_run": semua_fitness
            },
            "total_kurir": len(best_run['rute_mentah']),
            "waktu_komputasi_detik_terbaik": best_run['runtime_detik'],
            "total_waktu_semua_menit": round(best_run['total_waktu'], 2),
            "total_jarak_semua_km": round(best_run['total_jarak'], 2),
            "riwayat_konvergensi": best_run['riwayat_konvergensi'],
            "rute_per_kurir": []
        }
        
        for idx, kurir in enumerate(best_run['rute_mentah']):
            urutan_nama = [nama_depot]
            koordinat_list = [self.node_info[nama_depot]["koordinat"]]
            
            for node_idx in kurir['rute']:
                nama_rs = self.nodes[node_idx]
                urutan_nama.append(nama_rs)
                koordinat_list.append(self.node_info[nama_rs]["koordinat"])
                
            urutan_nama.append(nama_depot)
            koordinat_list.append(self.node_info[nama_depot]["koordinat"])
            
            klaster_data["rute_per_kurir"].append({
                "id_kurir": idx + 1,
                "waktu_tempuh_menit": round(kurir['waktu'], 2),
                "jarak_tempuh_km": round(kurir['jarak'], 2),
                "urutan_kunjungan": urutan_nama,
                "koordinat_kunjungan": koordinat_list
            })
            
        return klaster_data

In [5]:
# ==========================================
# 4. EKSEKUSI SEMUA KLASTER, PREVIEW, & GABUNG JSON
# ==========================================
print(f"🚀 MEMULAI PROSES GA VRP ({NUM_RUNS}x Run per Klaster)...\n")

# Keranjang besar penampung semua klaster (Nested Structure)
hasil_gabungan = {
    "algoritma": "Genetic Algorithm (GA)",
    "hasil_per_klaster": {}
}

# Jalankan GA untuk masing-masing klaster
for klaster in KLASTER_LIST:
    print("="*70)
    print(f"📍 MEMPROSES KLASTER {klaster.upper()}")
    print("-" * 70)
    
    ga = GAVRP(klaster)
    # Eksekusi fungsi multiple runs
    hasil_klaster = ga.run_multiple(NUM_RUNS)
    
    # --- PREVIEW TERMINAL ---
    stats = hasil_klaster['statistik_10_run']
    print("-" * 70)
    print(f"✅ HASIL TERBAIK KLASTER {klaster.upper()}:")
    print(f"Total Kurir          : {hasil_klaster['total_kurir']} Orang")
    print(f"Fitness Min (Jarak)  : {stats['fitness_minimum']} KM")
    print(f"Fitness Rata-Rata    : {stats['fitness_rata_rata']} KM")
    print(f"Standar Deviasi      : {stats['fitness_std_dev']}")
    print(f"Waktu Komputasi (Avg): {stats['waktu_komputasi_rata_rata_detik']} Detik")
    print("-" * 70)
    
    for kurir in hasil_klaster['rute_per_kurir']:
        print(f"🚚 [KURIR {kurir['id_kurir']}] - Jarak: {kurir['jarak_tempuh_km']} KM | Waktu: {kurir['waktu_tempuh_menit']} Menit")
        
        # Singkat nama di preview agar enak dibaca
        rute_singkat = " ➔ ".join([
            nama.replace("Puskesmas ", "P. ").replace("Pustu ", "P. ") 
            for nama in kurir['urutan_kunjungan']
        ])
        print(f"   Rute: {rute_singkat}\n")
    
    # Masukkan data ke keranjang besar dengan Key nama wilayah
    hasil_gabungan["hasil_per_klaster"][klaster.capitalize()] = hasil_klaster

# --- EXPORT JSON BERSARANG ---
print("="*70)
file_output = f"{PATH_JSON}rute_ga_gabungan.json"
with open(file_output, 'w') as f:
    json.dump(hasil_gabungan, f, indent=4)

print(f"🎉 SEMUA KLASTER SELESAI! Data statistik {NUM_RUNS} Run berhasil digabung ke file: {file_output}")

🚀 MEMULAI PROSES GA VRP (10x Run per Klaster)...

📍 MEMPROSES KLASTER BARAT
----------------------------------------------------------------------
      ➔ Run 1/10 Selesai | Fitness: 192.53 | Waktu: 7.639 dtk
      ➔ Run 2/10 Selesai | Fitness: 192.95 | Waktu: 8.502 dtk
      ➔ Run 3/10 Selesai | Fitness: 204.27 | Waktu: 18.561 dtk
      ➔ Run 4/10 Selesai | Fitness: 207.52 | Waktu: 32.014 dtk
      ➔ Run 5/10 Selesai | Fitness: 195.73 | Waktu: 32.456 dtk
      ➔ Run 6/10 Selesai | Fitness: 203.74 | Waktu: 32.889 dtk
      ➔ Run 7/10 Selesai | Fitness: 206.92 | Waktu: 32.696 dtk
      ➔ Run 8/10 Selesai | Fitness: 194.47 | Waktu: 32.316 dtk
      ➔ Run 9/10 Selesai | Fitness: 197.74 | Waktu: 32.362 dtk
      ➔ Run 10/10 Selesai | Fitness: 193.37 | Waktu: 26.333 dtk
----------------------------------------------------------------------
✅ HASIL TERBAIK KLASTER BARAT:
Total Kurir          : 2 Orang
Fitness Min (Jarak)  : 192.53 KM
Fitness Rata-Rata    : 198.92 KM
Standar Deviasi      : 6.

In [6]:
# --- EXPORT JSON BERSARANG ---
print("="*70)
file_output = f"{PATH_JSON}rute_ga.json"
with open(file_output, 'w') as f:
    json.dump(hasil_gabungan, f, indent=4)

print(f"🎉 SEMUA KLASTER SELESAI! Data berhasil digabung ke file: {file_output}")

🎉 SEMUA KLASTER SELESAI! Data berhasil digabung ke file: ../output_json/rute_ga.json
